In [ ]:
import os
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import requests

TMDB_API_KEY = "cc0093b5ad09a876190e097a1c2c8e65"
base_url = "https://api.themoviedb.org/3"

OUTPUT_DIR = "../data/expanded-data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Make core.game_logic importable (same trick streamlit/app.py uses: put the
# streamlit/ directory itself on sys.path so `core` resolves as a top-level
# package). game_logic.py has no streamlit import, so this works standalone.
sys.path.append("../streamlit")
from core.game_logic import build_and_save

In [ ]:
"""Shared GET helper with retry + exponential backoff.

Retries on network errors and on HTTP 429 (rate limited), honoring TMDB's
Retry-After header when present. All API calls in this notebook go through
this so a burst of 429s during the threaded movie/detail fetches degrades
into a slowdown instead of silently dropped/failed requests.
"""

def request_with_retry(url, params, max_retries=5, backoff_base=1.0, timeout=10):
    for attempt in range(max_retries):
        try:
            response = requests.get(url, params=params, timeout=timeout)
        except requests.exceptions.RequestException:
            if attempt == max_retries - 1:
                raise
            time.sleep(backoff_base * (2 ** attempt))
            continue

        if response.status_code == 429:
            retry_after = response.headers.get('Retry-After')
            wait = float(retry_after) if retry_after else backoff_base * (2 ** attempt)
            print(f"Rate limited by TMDB, waiting {wait:.1f}s (attempt {attempt + 1}/{max_retries})")
            time.sleep(wait)
            continue

        response.raise_for_status()
        return response

    raise requests.exceptions.RequestException(f"Exceeded {max_retries} retries for {url}")

In [ ]:
"""API call to retrieve the top N actors directly from TMDB's /person/popular

endpoint. Results are sorted by TMDB's popularity score; only people whose
known_for_department is "Acting" and who have at least one English-language
credit are kept, matching the same filter used for the existing dataset.
"""

def get_top_actors(limit=5000):
    actors = []
    page = 1

    while len(actors) < limit:
        try:
            response = request_with_retry(f"{base_url}/person/popular", params={
                'api_key': TMDB_API_KEY,
                'page': page
            })
            response = response.json()
        except requests.exceptions.RequestException as e:
            print(f"Error: {e}")
            break

        results = response.get('results', [])
        if not results:
            break

        for person in results:
            is_english = any(item.get('original_language') == "en" for item in person.get('known_for', []))
            is_actor = person.get('known_for_department') == "Acting"

            if is_english and is_actor:
                actors.append({
                    'name': person['name'],
                    'tmdb_id': person['id']
                })

            if len(actors) >= limit:
                break

        print(f"Collected: {len(actors)} actors (Page {page})", end='\r')
        page += 1
        time.sleep(0.1)

    return pd.DataFrame(actors)


actor_list = get_top_actors(limit=5000)
actor_list

In [ ]:
"""Single API call to retrieve the list of movies an actor is in."""

def get_movies(person_id):
    url = f"{base_url}/person/{person_id}/movie_credits"

    try:
        response = request_with_retry(url, params={
            'api_key': TMDB_API_KEY
        })
        response = response.json()
    except requests.exceptions.RequestException as e:
        print(f"Error: {e}")
        return None

    cast_list = pd.DataFrame(response['cast'])
    columns = ['id', 'original_language', 'original_title']
    cast_list = cast_list[columns].rename(columns={'id': 'movie_id'})

    return cast_list

In [ ]:
"""Executes get_movies() for the full list of actors.

max_workers is kept modest (rather than firing hundreds of requests at once)
so the pool of concurrent requests stays reasonable; request_with_retry
handles any 429s that still slip through.
"""

def get_all_movies(actor_ids, max_workers=8):
    full_list = []
    total = len(actor_ids)
    completed = 0

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(get_movies, actor_id): actor_id for actor_id in actor_ids}

        for future in as_completed(futures):
            actor_id = futures[future]
            completed += 1

            try:
                result = future.result()
                if result is not None and not result.empty:
                    result['actor_id'] = actor_id
                    full_list.append(result)
                    print(f"[{completed}/{total}] Actor {actor_id} - {len(result)} movies found")
                else:
                    print(f"[{completed}/{total}] Actor {actor_id} - no results")

            except Exception as e:
                print(f"Actor ID: {actor_id} encountered an error: {e}")

    full_df = pd.concat(full_list, ignore_index=True)
    return full_df


expanded_movie_list = get_all_movies(actor_list['tmdb_id'], max_workers=8)

In [ ]:
movie_list = expanded_movie_list.groupby(['movie_id', 'original_language', 'original_title'])['actor_id']
movie_list = movie_list.apply(lambda x: ",".join(x.astype(str))).reset_index()
movie_list = movie_list.rename(columns={"actor_id": "actors"})
movie_list = movie_list[movie_list['actors'].str.contains(",")]
movie_list = movie_list[movie_list['original_language'] == "en"]
movie_list

In [ ]:
"""Single API call to get the remaining movie details needed."""

def get_details(movie_id):
    url = f"{base_url}/movie/{movie_id}"

    try:
        response = request_with_retry(url, params={
            'api_key': TMDB_API_KEY
        })
        response = response.json()

        return {
            "id": movie_id,
            "year": response.get("release_date", "")[:4],
            "revenue": response.get("revenue")
        }

    except requests.exceptions.RequestException as e:
        print(f"Error: {e}")
        return {"id": None, "year": None, "revenue": None}

In [ ]:
"""max_workers is kept modest here too, for the same reason as get_all_movies."""

def get_all_details(movie_ids, max_workers=10):
    full_data = []
    total = len(movie_ids)
    completed = 0

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(get_details, movie_id): movie_id for movie_id in movie_ids}

        for future in as_completed(futures):
            completed += 1

            try:
                result = future.result()
                full_data.append(result)
            except Exception as e:
                print(f"Error: {e}")

            print(f"[{completed}/{total}]", end='\r')

    df = pd.DataFrame(full_data, columns=["id", "year", "revenue"])
    df = df.sort_values("id").reset_index(drop=True)

    return df


final_movie_details = get_all_details(movie_list["movie_id"], max_workers=10)

In [ ]:
inflation_data = pd.read_excel("../data/SeriesReport-20260302132218_6cc5fa.xlsx", header=11)
inflation_data['Annual'] = inflation_data['Annual'].fillna(inflation_data['Jan'])
inflation_data = inflation_data[['Year', 'Annual']]
cpi = pd.Series(inflation_data.set_index('Year')['Annual'])

def adjust_for_inflation(value, from_year, to_year=2026, cpi=cpi):
    if from_year == to_year:
        return value

    years = range(from_year + 1, to_year + 1)
    multiplier = 1
    for year in years:
        rate = cpi[year] / 100
        multiplier = multiplier * (1 + rate)

    return round(value * multiplier, 2)

In [ ]:
final_movie_list = movie_list.merge(final_movie_details, left_on='movie_id', right_on='id', how="left")
final_movie_list = final_movie_list.drop(columns=["original_language", "id"])
final_movie_list = final_movie_list[final_movie_list["year"] != ""]
final_movie_list = final_movie_list[final_movie_list["year"].isna() == False]
final_movie_list['year'] = final_movie_list['year'].astype(int)
final_movie_list = final_movie_list[final_movie_list['year'] >= 1958]
final_movie_list["adjusted_revenue"] = final_movie_list.apply(lambda row: adjust_for_inflation(row['revenue'], row['year']), axis=1)
final_movie_list

In [ ]:
final_movie_list.to_parquet(f"{OUTPUT_DIR}/final_movies.parquet")
actor_list.to_parquet(f"{OUTPUT_DIR}/actor_list.parquet")

In [ ]:
"""Build game_data.pkl from the freshly saved parquet files via
core.game_logic.build_and_save(), which reads this pipeline's own column
names (movie_id/original_title/actors/year/adjusted_revenue and
tmdb_id/name) directly - no renaming or temp files needed.
"""

build_and_save(
    f"{OUTPUT_DIR}/final_movies.parquet",
    f"{OUTPUT_DIR}/actor_list.parquet",
    output_path=f"{OUTPUT_DIR}/game_data.pkl",
)